# ProfitTune – Modelo de predicción de éxito musical de género electrónica basado en ganancia económica

# **Autor:** Juan Cruz Cordoneda  

## Objetivo
El proyecto tiene como propósito predecir el **éxito comercial de canciones de género electrónica** mediante modelos de *Machine Learning* entrenados con variables de audio como tempo, energía, valencia, entre otras.

## Enfoque
En lugar de utilizar métricas tradicionales (precisión, recall o F1-score), se emplea una **función de ganancia económica** que traduce los resultados del modelo en términos monetarios, simulando escenarios reales de inversión.

## Métrica de Evaluación
Cada tipo de predicción tiene un valor económico asociado:

- **True Positive (TP):** +USD 5000  
- **False Positive (FP):** −USD 1000  
- **False Negative (FN):** −USD 2000  
- **True Negative (TN):** sin impacto económico  

El objetivo final es **maximizar la ganancia total (profit)**, ajustando dinámicamente el umbral de probabilidad para encontrar el punto óptimo entre riesgo y retorno.


In [30]:
# === Librerías básicas ===
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# === Visualización ===
import plotly.express as px
import plotly.graph_objects as go

# === Machine Learning ===
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import confusion_matrix

# === Preprocesamiento y validación ===
from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV
from sklearn.preprocessing import StandardScaler
from scipy.stats import randint, uniform


In [31]:
# Función de ganancia económica personalizada
def profit_score(y_true, y_pred, Ganancia_hit=5000, Costo_fp=1000, Costo_fn=2000, show_plots=True):
    """
    Calcula la ganancia total simulada y, si se desea, muestra:
    - Matriz de confusión (TP, FP, FN, TN)

    Retorna:
    --------
    profit : Ganancia total del modelo (USD)
    """

    # === Cálculo base ===
    tp = np.sum((y_true == 1) & (y_pred == 1))
    fp = np.sum((y_true == 0) & (y_pred == 1))
    fn = np.sum((y_true == 1) & (y_pred == 0))
    tn = np.sum((y_true == 0) & (y_pred == 0))
    profit = tp * Ganancia_hit - fp * Costo_fp - fn * Costo_fn

    if show_plots:
        # === Matriz de confusión ===
        fig_conf = go.Figure(data=go.Heatmap(
            z=[[tn, fp],
               [fn, tp]],
            x=["Predice No Éxito", "Predice Éxito"],
            y=["Real No Éxito", "Real Éxito"],
            colorscale="Blues",
            text=[[f"TN<br>{tn}", f"FP<br>{fp}"],
                  [f"FN<br>{fn}", f"TP<br>{tp}"]],
            texttemplate="%{text}",
            showscale=False
        ))
        fig_conf.update_layout(
            title="Matriz de Confusión (TP / FP / FN / TN)",
            xaxis_title="Predicción del modelo",
            yaxis_title="Valor real",
            template="plotly_white"
        )
        fig_conf.show()

    return profit


# === Nueva función de scoring con búsqueda de mejor umbral dinámico ===
def profit_scorer_dynamic(estimator, X, y_true):
    # Obtiene las probabilidades de predicción (columna de clase positiva)
    y_pred_proba = estimator.predict_proba(X)[:, 1]

    # Inicializa la mejor ganancia como menos infinito (para comparar)
    best_profit = -np.inf

    # Recorre posibles umbrales de decisión (de 0.01 a 0.49)
    for t in np.arange(0.01, 0.5, 0.01):
        # Convierte las probabilidades en predicciones binarias según el umbral
        y_pred = (y_pred_proba >= t).astype(int)

        # Calcula la ganancia económica para ese umbral
        profit = profit_score(y_true, y_pred, show_plots=False)

        # Si la ganancia mejora, actualiza el mejor valor encontrado
        if profit > best_profit:
            best_profit = profit

    # Devuelve la mejor ganancia obtenida entre todos los umbrales
    return best_profit


In [32]:
# === PASO 1: CARGA Y LIMPIEZA DEL DATASET ===

# Cargar dataset descargado de Kaggle
df = pd.read_csv("EDMHits.csv")

# Renombrar columnas a español
df.columns = [
    "Cancion", "Artista", "Album", "Año", "Duracion_ms", "Compas",
    "Bailabilidad", "Energia", "Tonalidad", "Volumen", "Modo", "Habla",
    "Acustica", "Instrumentalidad", "Presencia_en_vivo", "Valencia",
    "Tempo", "Popularidad"
]

# Eliminar columnas irrelevantes
df = df.drop(["Cancion", "Artista", "Album"], axis=1)

# Crear variable objetivo: Éxito
df["Exito"] = (df["Popularidad"] >= 70).astype(int)

df.head()


,Año,Duracion_ms,Compas,Bailabilidad,Energia,Tonalidad,Volumen,Modo,Habla,Acustica,Instrumentalidad,Presencia_en_vivo,Valencia,Tempo,Popularidad,Exito
0,2009,412266,4,0.650,0.816,0,-5.749,1,0.0476,0.004710,0.458000,0.0896,0.5190,128.012,9,0
1,2009,206400,4,0.506,0.822,3,-8.630,0,0.0497,0.255000,0.051600,0.0775,0.3700,137.909,33,0
2,2014,400226,4,0.691,0.581,5,-6.563,1,0.0386,0.000701,0.031100,0.0855,0.2650,130.014,16,0
3,2013,192866,4,0.641,0.932,7,-5.797,0,0.0658,0.065700,0.035300,0.0600,0.3350,132.991,38,0
4,2019,332946,4,0.482,0.525,1,-10.987,1,0.0408,0.089400,0.000507,0.0652,0.0867,125.985,34,0


In [33]:
# === PASO 2: ANÁLISIS EXPLORATORIO DE DATOS ===

# Distribución de popularidad
fig = px.histogram(df, x="Popularidad", color="Exito", nbins=20,
                   title="Distribución de popularidad según éxito")
fig.show()

# Mapa de correlaciones
corr = df.drop("Exito", axis=1).corr()
fig = px.imshow(corr, text_auto=True, title="Mapa de Correlaciones")
fig.show()


In [34]:
# === PASO 3: DIVISIÓN TRAIN Y TEST ===

# Separar variables
X = df.drop(["Exito"], axis=1)
y = df["Exito"]

# Escalado
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# División entrenamiento / test
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.3, random_state=42, stratify=y)


In [35]:
# === PASO 4: MODELO RANDOM FOREST BASE ===

rf = RandomForestClassifier(random_state=42)
rf.fit(X_train, y_train)
y_pred_rf = (rf.predict_proba(X_test)[:,1] >= 0.09).astype(int)
ganancia_rf_base = profit_score(y_test, y_pred_rf)
print("Ganancia Random Forest base:", ganancia_rf_base)


Ganancia Random Forest base: 98000


In [36]:
# === PASO 5: BUSQUEDA DE HIPERPARÁMETROS RANDOM FOREST ===

param_grid = {
    "n_estimators": [200, 400, 600],    #Controla cuántos árboles componen el bosque.
    "max_depth": [8, 12, 16, None],     #Profundidad máxima de cada árbol.
    "min_samples_split": [2, 5, 10],    #Número mínimo de muestras necesarias para dividir un nodo.
    "min_samples_leaf": [1, 2, 4],      #Número mínimo de muestras necesarias en un nodo hoja.
    "max_features": ["sqrt", "log2"]    #Define cuántas variables se prueban al dividir cada nodo.
}

grid_search = GridSearchCV(
    estimator=RandomForestClassifier(random_state=42),      # Modelo base: Random Forest con semilla fija para reproducibilidad
    param_grid=param_grid,                                  # Parámetros a probar en la búsqueda    
    scoring=profit_scorer_dynamic,                          # Métrica personalizada para evaluar (usa ganancia económica)
    cv=3,
    n_jobs=-1,
    verbose=1
)


grid_search.fit(X_train, y_train)
best_model = grid_search.best_estimator_

print("\nMejores parámetros encontrados:")
print(grid_search.best_params_)
print(best_model)
print(f"Mejor ganancia promedio (3 folds): {grid_search.best_score_:.2f}")

# === Evaluar el mejor modelo ===

# Obtiene las probabilidades del mejor Random Forest sobre el set de test
y_pred_proba_best = best_model.predict_proba(X_test)[:, 1]

# Diccionario para guardar la ganancia en cada umbral
profits = {}

# Recorre umbrales de decisión del 0.01 al 0.49
for t in np.arange(0.01, 0.5, 0.01):
    # Convierte probabilidades en predicciones binarias según el umbral
    y_pred_t = (y_pred_proba_best >= t).astype(int)

    # Calcula la ganancia económica del modelo con ese umbral
    profits[t] = profit_score(y_test, y_pred_t, show_plots=False)

best_threshold = max(profits, key=profits.get)
best_profit = profits[best_threshold]

print(f"\nMejor umbral en test: {best_threshold:.2f}")
print(f"Ganancia máxima en test: {best_profit:.2f}")


Fitting 3 folds for each of 216 candidates, totalling 648 fits

Mejores parámetros encontrados:
{'max_depth': 8, 'max_features': 'sqrt', 'min_samples_leaf': 1, 'min_samples_split': 2, 'n_estimators': 200}
RandomForestClassifier(max_depth=8, n_estimators=200, random_state=42)
Mejor ganancia promedio (3 folds): 86666.67

Mejor umbral en test: 0.22
Ganancia máxima en test: 110000.00


In [37]:
# === PASO 6: MODELO FINAL RANDOM FOREST OPTIMIZADO ===

# Parámetros óptimos encontrados en el GridSearch
best_params = {
    "n_estimators": 200,
    "max_depth": 8,
    "max_features": "sqrt",
    "min_samples_leaf": 1,
    "min_samples_split": 2,
    "random_state": 42
}

# === Entrenar el modelo final con los mejores hiperparámetros ===
rf_final = RandomForestClassifier(**best_params)
rf_final.fit(X_train, y_train)

# === Predecir probabilidades sobre el conjunto de test ===
y_pred_proba_final = rf_final.predict_proba(X_test)[:, 1]

# === Usar el mejor umbral encontrado (0.22) ===
best_threshold = 0.22
y_pred_final = (y_pred_proba_final >= best_threshold).astype(int)

# === Calcular la ganancia final ===
ganancia_final = profit_score(y_test, y_pred_final)

print("MODELO FINAL RANDOM FOREST OPTIMIZADO")
print(f"Parámetros: {best_params}")
print(f"Umbral de decisión: {best_threshold}")
print(f"Ganancia final en test: {ganancia_final:.2f}")


MODELO FINAL RANDOM FOREST OPTIMIZADO
Parámetros: {'n_estimators': 200, 'max_depth': 8, 'max_features': 'sqrt', 'min_samples_leaf': 1, 'min_samples_split': 2, 'random_state': 42}
Umbral de decisión: 0.22
Ganancia final en test: 110000.00


In [38]:
# === PASO 7: MODELO XGBOOST ===

xgb = XGBClassifier(
    n_estimators=300,
    max_depth=5,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)

xgb.fit(X_train, y_train)
y_pred_xgb = (xgb.predict_proba(X_test)[:, 1] >= 0.09).astype(int)
ganancia_xgb = profit_score(y_test, y_pred_xgb)
print("Ganancia XGBoost:", ganancia_xgb)


Ganancia XGBoost: 110000


In [39]:
# === PASO 8: COMPARACIÓN DE GANANCIAS ===

ganancias = {
    "Random Forest Base": ganancia_rf_base,
    "Random Forest Optimizado": ganancia_final,
    "XGBoost": best_profit
}

fig = go.Figure(data=[
    go.Bar(
        x=list(ganancias.keys()),
        y=list(ganancias.values()),
        text=[f"${v:,.0f}" for v in ganancias.values()],
        textposition="auto"
    )
])

fig.update_layout(
    title="Comparación de Ganancias por Modelo",
    xaxis_title="Modelo",
    yaxis_title="Ganancia (USD)",
    template="plotly_white"
)

fig.show()



# Conclusiones Finales

El proyecto aplicó técnicas de Ciencia de Datos para predecir el éxito comercial de canciones electrónicas usando atributos sonoros de Spotify.

Los resultados mostraron que los modelos **Random Forest optimizado y XGBoost** alcanzaron la mayor rentabilidad, con una ganancia simulada de aproximadamente **USD 110.000**, superando ampliamente al modelo Random Forest  base, que obtuvo una ganancia más modesta (~USD 98.000).

Las variables más influyentes fueron **energía**, **valencia** y **bailabilidad**, confirmando que los temas alegres, enérgicos y con buen ritmo tienden a tener mayor impacto comercial.

El enfoque económico de evaluación (basado en ganancia monetaria) resultó más útil que las métricas clásicas de precisión, mostrando cómo la IA puede transformar decisiones de inversión en la industria musical.